# Ejemplo de uso del ORM IGSM

Este notebook muestra cómo usar la API pública del ORM con una base SQLite temporal, sin tocar `database/igsm_dev.sqlite3` ni ninguna base configurada en `DATABASE_URL`.

La idea es ejemplificar las estructuras de retorno y reutilizar el patrón en la futura implementación de la aplicación.


## 1. Preparar imports y base temporal

La celda busca la raíz del proyecto, agrega esa ruta a `sys.path` y crea una URL SQLite dentro de un directorio temporal.


In [1]:
from datetime import date
from pathlib import Path
import sys
from uuid import uuid4

from IPython.display import display
import pandas as pd
from sqlalchemy import func, inspect, select


def find_project_root(start: Path) -> Path:
    """Find the repository root from a starting directory.

    Args:
        start: Directory where the search starts.

    Returns:
        Repository root containing `database/` and `data/`.

    Raises:
        RuntimeError: If the repository root cannot be found.
    """

    for candidate in (start, *start.parents):
        if (candidate / "database").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("No se encontró la raíz del proyecto con database/ y data/.")


ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from database.models import (  # noqa: E402
    Base,
    DMAxis,
    DMIndicator,
    DMMunicipality,
    DMService,
    FactIndicatorResponse,
    FactMaturityThreshold,
    FactStageWeight,
)
from database.repositories import (  # noqa: E402
    get_indicators_for_service,
    get_latest_indicator_responses,
    get_latest_maturity_thresholds,
    get_latest_responses_for_municipality,
    get_latest_stage_weights,
    get_municipality_completion_statistics,
    get_national_statistics,
    get_services_for_municipality,
    list_municipalities,
    list_municipalities_by_region,
    submit_indicator_responses,
)
from database.seed import seed_all  # noqa: E402
from database.session import get_engine, session_scope  # noqa: E402


TEMP_PARENT = ROOT / ".test_tmp"
TEMP_PARENT.mkdir(exist_ok=True)
DB_PATH = TEMP_PARENT / f"ejemplo_orm_{uuid4().hex}.sqlite3"
DATABASE_URL = f"sqlite:///{DB_PATH.as_posix()}"
END_DATE = date(2025, 1, 1)
REAL_END_DATE = date(2025, 12, 31)
engine = get_engine(DATABASE_URL)

print(f"Raíz del proyecto: {ROOT}")
print(f"Base temporal: {DB_PATH}")


Raíz del proyecto: C:\Users\jason\PycharmProjects\Projects Ulead\Proyecto_Analisis_de_datos_Grupo-2
Base temporal: C:\Users\jason\PycharmProjects\Projects Ulead\Proyecto_Analisis_de_datos_Grupo-2\.test_tmp\ejemplo_orm_c497f3609b974dada129f7335b1ce9aa.sqlite3


## 2. Crear el esquema y sembrar datos

`Base.metadata.create_all(engine)` crea las tablas definidas por los modelos ORM. <br> `seed_all(session=session)` carga los datos de referencia necesarios para trabajar con municipalidades, servicios, indicadores, pesos y umbrales.


In [2]:
Base.metadata.create_all(engine)

with session_scope(DATABASE_URL) as session:
    seed_all(session=session)
    counts = {
        "municipalidades": session.scalar(select(func.count()).select_from(DMMunicipality)),
        "ejes": session.scalar(select(func.count()).select_from(DMAxis)),
        "servicios": session.scalar(select(func.count()).select_from(DMService)),
        "indicadores": session.scalar(select(func.count()).select_from(DMIndicator)),
        "pesos_etapa": session.scalar(select(func.count()).select_from(FactStageWeight)),
        "umbrales_madurez": session.scalar(select(func.count()).select_from(FactMaturityThreshold)),
    }

tables_df = pd.DataFrame({"tabla": sorted(inspect(engine).get_table_names())})
counts_df = pd.DataFrame(
    [{"conjunto": name, "filas": value} for name, value in counts.items()]
)

print("Tablas creadas:")
display(tables_df)

print("Conteos de datos de referencia:")
display(counts_df)


Tablas creadas:


,tabla
0,dm_axis
1,dm_indicator
2,dm_municipality
3,dm_municipality_diversified_service
4,dm_service
5,dm_stage
6,fact_indicator_response
7,fact_maturity_threshold
8,fact_stage_weight


Conteos de datos de referencia:


,conjunto,filas
0,municipalidades,84
1,ejes,3
2,servicios,10
3,indicadores,159
4,pesos_etapa,1
5,umbrales_madurez,1


## 3. Consultar municipalidades

Las funciones de lectura devuelven diccionarios listos para la interfaz. <br> 
*  `list_municipalities` trae todas las municipalidades 
* `list_municipalities_by_region` filtra por región.


In [3]:
with session_scope(DATABASE_URL) as session:
    municipalities = list_municipalities(session=session)
    sample_municipality = municipalities[0]
    same_region = list_municipalities_by_region(sample_municipality["region"], session=session)

municipality_df = pd.DataFrame([sample_municipality])
same_region_df = pd.DataFrame(same_region)

print("Municipalidad de ejemplo:")
display(municipality_df)

print(f"Municipalidades en la región {sample_municipality['region']}: {len(same_region_df)}")
display(same_region_df[["codigo", "nombre", "provincia", "region", "diversificados"]].head(10))


Municipalidad de ejemplo:


,codigo,nombre,municipalidad,provincia,region,lat,lon,diversificados
0,507,Abangares,Abangares,Guanacaste,Chorotega,10.2928,-85.0267,[]


Municipalidades en la región Chorotega: 11


,codigo,nombre,provincia,region,diversificados
0,507,Abangares,Guanacaste,Chorotega,[]
1,504,Bagaces,Guanacaste,Chorotega,[]
2,505,Carrillo,Guanacaste,Chorotega,[zmt]
3,506,Cañas,Guanacaste,Chorotega,[]
4,511,Hojancha,Guanacaste,Chorotega,[zmt]
5,510,La Cruz,Guanacaste,Chorotega,[]
6,501,Liberia,Guanacaste,Chorotega,"[agua_potable, seguridad, zmt]"
7,509,Nandayure,Guanacaste,Chorotega,[zmt]
8,502,Nicoya,Guanacaste,Chorotega,[zmt]
9,503,Santa Cruz,Guanacaste,Chorotega,[zmt]


## 4. Consultar servicios, indicadores, pesos y umbrales

* `get_services_for_municipality` combina servicios básicos con los diversificados aplicables a la municipalidad. 
* `get_indicators_for_service` devuelve los indicadores de un servicio específico.


In [4]:
with session_scope(DATABASE_URL) as session:
    services = get_services_for_municipality(sample_municipality["codigo"], session=session)
    first_service = next(iter(services.values()))
    indicators = get_indicators_for_service(first_service["service_code"], session=session)
    weights = get_latest_stage_weights(end_date=END_DATE, session=session)
    thresholds = get_latest_maturity_thresholds(end_date=END_DATE, session=session)

services_df = pd.DataFrame(services.values()).sort_values("service_code")
indicators_df = pd.DataFrame(indicators)
weights_df = pd.DataFrame(
    [{"etapa": stage, "peso": weight} for stage, weight in weights.items()]
)
thresholds_df = pd.DataFrame(
    [{"umbral": name, "valor": value} for name, value in thresholds.items()]
)

print(f"Servicios aplicables para {sample_municipality['nombre']}: {len(services_df)}")
display(services_df[["service_code", "name", "grouping", "axis", "diversificado_key"]])

print(f"10 Indicadores del servicio '{first_service['name']}':")
display(indicators_df[["codigo", "nombre", "etapa", "tipo", "evidencia"]].head(10))

print(f"Pesos de etapa vigentes al {END_DATE.isoformat()}:")
display(weights_df)

print(f"Umbrales de madurez vigentes al {END_DATE.isoformat()}:")
display(thresholds_df)


Servicios aplicables para Abangares: 7


,service_code,name,grouping,axis,diversificado_key
0,1.1.1,"Recolección, depósito y tratamiento de residuo...",Básico,Salubridad Pública,None
1,1.1.2,Aseo de vías y sitios públicos,Básico,Salubridad Pública,None
2,1.2.1,Urbanismo e Infraestructura,Básico,Desarrollo Urbano,None
3,1.2.2,Red Vial Cantonal,Básico,Desarrollo Urbano,None
4,1.2.3,Alcantarillado pluvial,Básico,Desarrollo Urbano,None
5,1.3.1,Servicios Sociales y complementarios,Básico,Servicios Sociales,None
6,1.3.2,"Educativos, culturales y deportivos",Básico,Servicios Sociales,None


10 Indicadores del servicio 'Recolección, depósito y tratamiento de residuos sólidos':


,codigo,nombre,etapa,tipo,evidencia
0,1.1.1.1,"Se brinda el servicio de recolección, depósito...",Planificación,None,None
1,1.1.1.10,"Cobertura del servicio de recolección, depósit...",Ejecución,None,None
2,1.1.1.11,Recolección de residuos valorizables,Ejecución,None,None
3,1.1.1.12,Cobertura del Servicio de Recolección de Resid...,Ejecución,None,None
4,1.1.1.13,Porcentaje de valorización en el servicio,Ejecución,None,None
5,1.1.1.14,Sensibilización de la ciudadanía en respuesta ...,Ejecución,None,None
6,1.1.1.15,Recursos del servicio destinados por distrito,Ejecución,None,None
7,1.1.1.16,Ingresos y gastos del servicio (Indicador de d...,Ejecución,None,None
8,1.1.1.17,Nivel de ejecución de los recursos disponibles...,Ejecución,None,None
9,1.1.1.18,Recursos destinados al desarrollo del servicio,Ejecución,None,None


Pesos de etapa vigentes al 2025-01-01:


,etapa,peso
0,Planificación,0.5
1,Ejecución,0.3
2,Evaluación,0.2


Umbrales de madurez vigentes al 2025-01-01:


,umbral,valor
0,initial_upper,0.31
1,basic_upper,0.56
2,intermediate_upper,0.76
3,advanced_upper,0.91
4,optimizing_upper,1.00


## 5. Guardar respuestas de indicadores

* `submit_indicator_responses` recibe el código de municipalidad, una fecha de respuesta `end_date` y un diccionario `{codigo_indicador: valor}`.
* El ORM guarda esos datos directamente en `fact_indicator_response` y devuelve solo un resumen de persistencia.


In [5]:
with session_scope(DATABASE_URL) as session:
    responses = {indicator["codigo"]: 1.0 for indicator in indicators[:8]}
    submission = submit_indicator_responses(
        sample_municipality["codigo"],
        END_DATE,
        responses,
        session=session,
    )

submission_df = pd.DataFrame(
    [
        {
            "codigo_municipalidad": submission["municipality_code"],
            "end_date": submission["end_date"],
            "respuestas_guardadas": submission["responses_count"],
            "hechos_guardados": len(submission["fact_response_ids"]),
        }
    ]
)
fact_ids_df = pd.DataFrame({"fact_response_id": submission["fact_response_ids"]})

print("Resumen de submit_indicator_responses:")
display(submission_df)

print("IDs creados o actualizados en fact_indicator_response:")
display(fact_ids_df)


Resumen de submit_indicator_responses:


,codigo_municipalidad,end_date,respuestas_guardadas,hechos_guardados
0,507,2025-01-01,8,8


IDs creados o actualizados en fact_indicator_response:


,fact_response_id
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8


## 6. Leer respuestas y completitud

Después de guardar hechos, el ORM permite leer respuestas y medir completitud de datos. Los scores, niveles y rankings se calculan fuera de esta capa, en `data/calculation.py`.


In [6]:
with session_scope(DATABASE_URL) as session:
    latest = get_latest_responses_for_municipality(
        sample_municipality["codigo"],
        end_date=END_DATE,
        session=session,
    )
    all_latest = get_latest_indicator_responses(end_date=END_DATE, session=session)
    stats = get_national_statistics(end_date=END_DATE, session=session)
    municipality_stats = get_municipality_completion_statistics(
        sample_municipality["codigo"],
        end_date=END_DATE,
        session=session,
    )

latest_df = pd.DataFrame(
    [{"codigo_indicador": code, "valor": value} for code, value in latest.items()]
)
all_latest_df = pd.DataFrame(all_latest)
stats_df = pd.DataFrame([{"metrica": key, "valor": value} for key, value in stats.items()])
municipality_stats_df = pd.DataFrame(
    [{"metrica": key, "valor": value} for key, value in municipality_stats.items()]
)

print("Respuestas más recientes para la municipalidad de muestra:")
display(latest_df)

print("Todas las respuestas más recientes a la fecha de corte:")
display(all_latest_df.head(20))

print("Completitud nacional:")
display(stats_df)

print("Completitud municipal:")
display(municipality_stats_df)


Respuestas más recientes para la municipalidad de muestra:


,codigo_indicador,valor
0,1.1.1.16,1.0
1,1.1.1.15,1.0
2,1.1.1.14,1.0
3,1.1.1.13,1.0
4,1.1.1.12,1.0
5,1.1.1.11,1.0
6,1.1.1.10,1.0
7,1.1.1.1,1.0


Todas las respuestas más recientes a la fecha de corte:


,response_id,fecha_respuesta,codigo_municipalidad,municipalidad,codigo_indicador,indicador,valor
0,1,2025-01-01,507,Abangares,1.1.1.1,"Se brinda el servicio de recolección, depósito...",1.0
1,2,2025-01-01,507,Abangares,1.1.1.10,"Cobertura del servicio de recolección, depósit...",1.0
2,3,2025-01-01,507,Abangares,1.1.1.11,Recolección de residuos valorizables,1.0
3,4,2025-01-01,507,Abangares,1.1.1.12,Cobertura del Servicio de Recolección de Resid...,1.0
4,5,2025-01-01,507,Abangares,1.1.1.13,Porcentaje de valorización en el servicio,1.0
5,6,2025-01-01,507,Abangares,1.1.1.14,Sensibilización de la ciudadanía en respuesta ...,1.0
6,7,2025-01-01,507,Abangares,1.1.1.15,Recursos del servicio destinados por distrito,1.0
7,8,2025-01-01,507,Abangares,1.1.1.16,Ingresos y gastos del servicio (Indicador de d...,1.0


Completitud nacional:


,metrica,valor
0,end_date,2025-01-01
1,total_municipalidades,84
2,total_indicadores,159
3,respuestas_esperadas,13356
4,respuestas_recibidas,8
5,pct_completitud,0.06


Completitud municipal:


,metrica,valor
0,end_date,2025-01-01
1,codigo,507
2,municipalidad,Abangares
3,total_indicadores,159
4,respuestas_esperadas,159
5,respuestas_recibidas,8
6,pct_completitud,5.03


### 6.1 Lectura del dataset real en modo read-only

Esta celda abre `database/igsm_dev.sqlite3` en modo solo lectura cuando el archivo existe. Sirve para repetir la prueba de completitud contra el dataset real sin escribir hechos ni modificar la base.


In [7]:
REAL_DB_PATH = ROOT / "database" / "igsm_dev.sqlite3"

if REAL_DB_PATH.exists():
    read_only_url = f"sqlite:///file:{REAL_DB_PATH.as_posix()}?mode=ro&uri=true"
    read_only_engine = get_engine(read_only_url)
    try:
        with session_scope(read_only_url) as session:
            real_municipality = list_municipalities(session=session)[0]
            real_latest = get_latest_responses_for_municipality(
                real_municipality["codigo"],
                end_date=REAL_END_DATE,
                session=session,
            )
            real_all_latest = get_latest_indicator_responses(end_date=REAL_END_DATE, session=session)
            real_stats = get_national_statistics(end_date=REAL_END_DATE, session=session)
            real_municipality_stats = get_municipality_completion_statistics(
                real_municipality["codigo"],
                end_date=REAL_END_DATE,
                session=session,
            )

        print(f"Base real abierta en solo lectura: {REAL_DB_PATH}")
        print(f"Fecha de corte: {REAL_END_DATE.isoformat()}")
        print(f"Municipalidad de muestra: {real_municipality['nombre']} ({real_municipality['codigo']})")
        print(f"Respuestas leídas para la municipalidad: {len(real_latest)}")
        print(f"Respuestas más recientes nacionales: {len(real_all_latest)}")
        display(pd.DataFrame([real_stats]))
        display(pd.DataFrame([real_municipality_stats]))
    finally:
        read_only_engine.dispose()
else:
    print(f"No se encontró base real para lectura: {REAL_DB_PATH}")


Base real abierta en solo lectura: C:\Users\jason\PycharmProjects\Projects Ulead\Proyecto_Analisis_de_datos_Grupo-2\database\igsm_dev.sqlite3
Fecha de corte: 2025-12-31
Municipalidad de muestra: Abangares (507)
Respuestas leídas para la municipalidad: 103
Respuestas más recientes nacionales: 8840


,end_date,total_municipalidades,total_indicadores,respuestas_esperadas,respuestas_recibidas,pct_completitud
0,2025-12-31,85,159,13515,8840,65.41


,end_date,codigo,municipalidad,total_indicadores,respuestas_esperadas,respuestas_recibidas,pct_completitud
0,2025-12-31,507,Abangares,159,159,103,64.78


## 7. Limpieza

Al terminar, se cierra el engine y se elimina siempre la base SQLite temporal creada para este notebook, junto con posibles archivos laterales de SQLite.


In [8]:
engine.dispose()

for suffix in ("", "-wal", "-shm", "-journal"):
    db_file = DB_PATH if suffix == "" else DB_PATH.with_name(f"{DB_PATH.name}{suffix}")
    db_file.unlink(missing_ok=True)

print(f"Base temporal eliminada: {DB_PATH}")


Base temporal eliminada: C:\Users\jason\PycharmProjects\Projects Ulead\Proyecto_Analisis_de_datos_Grupo-2\.test_tmp\ejemplo_orm_c497f3609b974dada129f7335b1ce9aa.sqlite3
